In [64]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [65]:
llm = ChatOllama(model="llama3.2", temperature=0.2)



poem = llm.invoke("Write me a 4 line poem for my love for samosa")
print(poem.content)

Here's a short poem for your love of samosas:

Golden pastry, crispy delight
Samosa's flavors dance in the night
Spiced potatoes and peas entwined
My heart beats for this savory find


In [66]:
import json

##with open("../data/mitre-attack/enterprise-attack.json", encoding="utf-8") as f:
  ##  data = json.load(f)


##print(type(data))
#print(data.keys())
#print(f"Total objects: {len(data['objects'])}")

In [67]:
#for obj in data["objects"]:
#           if obj["type"] == "attack-pattern":
#               print(json.dumps(obj, indent=2)[:2000])
#               break

In [68]:
import pandas as pd

INPUT_PATH = "../data/mitre-attack/enterprise-attack.json"
OUTPUTPATH = "../data/mitre_techniques.csv"

def load_attack_data():
    with open(INPUT_PATH, encoding="utf-8") as f:
        return json.load(f)

In [69]:
data = load_attack_data()
print(len(data["objects"]))

26085


In [70]:
def get_technique_id(technique):
    for ref in technique.get("external_references", []):
        if ref.get("source_name") == "mitre-attack":
            return ref.get("external_id")
    return None

In [71]:
def parse_techniques(bundle):
    rows = []
    for obj in bundle["objects"]:
        if obj["type"] != "attack-pattern":
            continue
        if obj.get("revoked") or obj.get("x_mitre_deprecated") == True:
            continue

        tactics = [phase["phase_name"] for phase in obj.get("kill_chain_phases", [])]

        rows.append({
            "technique_id": get_technique_id(obj),
            "name": obj["name"],
            "tactics": ", ".join(tactics),
            "platforms": ", ".join(obj.get("x_mitre_platforms", [])),
            "description": clean_description(obj.get("description", "")),
        })
        
    return rows


rows = parse_techniques(load_attack_data())
print(len(rows))

697


In [72]:
df = pd.DataFrame(rows)
df.to_csv(OUTPUTPATH, index=False, encoding="utf-8")
print(f"Saved {len(df)} techniques")

Saved 697 techniques


#### Cleaning the data

In [73]:
import re

def clean_description(text):
    text = re.sub(r"\(Citation: [^)]*\)", "", text)
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    text = re.sub(r"</?code>", "", text)
    text = re.sub(r" +", " ", text)
    return text.strip()

In [74]:
before = rows[0]["description"]
print(before[:500])
print("-----")
print(clean_description(before)[:500])

Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data). Regist
-----
Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows class, which stipulate appearance and behavior (via windows procedures, which are functions that handle input/output of data).

In [75]:
print(df["description"].str.contains("Citation").sum())

0


In [76]:

CSV_PATH = "../data/mitre_techniques.csv"

loader = CSVLoader(file_path=CSV_PATH, source_column="technique_id", encoding="utf-8")
docs = loader.load()

print(len(docs))
print(docs[0].page_content[:500])
print(docs[0].metadata)

697
technique_id: T1055.011
name: Extra Window Memory Injection
tactics: stealth, privilege-escalation
platforms: Windows
description: Adversaries may inject malicious code into process via Extra Window Memory (EWM) in order to evade process-based defenses as well as possibly elevate privileges. EWM injection is a method of executing arbitrary code in the address space of a separate live process. 

Before creating a window, graphical Windows-based processes must prescribe to or register a windows cl
{'source': 'T1055.011', 'row': 0}


In [77]:
print(df["technique_id"].isna().sum())

0


In [78]:

VECTORDB_PATH = "../faiss_index"

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



vectordb = FAISS.from_documents(documents=docs, embedding=embeddings)

vectordb.save_local(VECTORDB_PATH)
print("Vector database saved!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database saved!


In [79]:
results = vectordb.similarity_search("powershell encoded command hidden window", k=4)
for doc in results:
    print(doc.metadata["source"], "-", doc.page_content.split("\n")[1])

print("-----")

results = vectordb.similarity_search(
    "powershell.exe -nop -w hidden -enc SQBFAFgAIAAoAE4AZQB3AC0ATwBiAGoAZQBjAHQA", k=4
)
for doc in results:
    print(doc.metadata["source"], "-", doc.page_content.split("\n")[1])

T1027.003 - name: Steganography
T1564.003 - name: Hidden Window
T1027.010 - name: Command Obfuscation
T1059.001 - name: PowerShell
-----
T1027.003 - name: Steganography
T1216.002 - name: SyncAppvPublishingServer
T1059.001 - name: PowerShell
T1564.003 - name: Hidden Window
